In [17]:
import os
import ffmpeg
import subprocess
from pathlib import Path
import numpy as np

In [18]:
input_path = "./input/naruto_clip.mp4"
output_filename = "output_naruto_clip"

if os.path.exists(input_path):
    print(f"Exists: {input_path}")
else:
    print(f"Input path does not exist: {input_path}")


Exists: ./input/naruto_clip.mp4


In [19]:
BASE_DIR = Path.cwd().resolve()
print(f"BASE_DIR: {BASE_DIR}")

OUTPUT_DIR = BASE_DIR / "output"
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

TEMP_DIR = BASE_DIR / "temp"
print(f"TEMP_DIR: {TEMP_DIR}")

BASE_DIR: C:\Dev\AMVerge\backend\notebooks\ffmpeg_practice
OUTPUT_DIR: C:\Dev\AMVerge\backend\notebooks\ffmpeg_practice\output
TEMP_DIR: C:\Dev\AMVerge\backend\notebooks\ffmpeg_practice\temp


In [20]:
BASE_DIR = Path.cwd().resolve()
INPUT_DIR = BASE_DIR / "input"
INPUT_FILE = INPUT_DIR / "naruto_clip.mp4"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_FILE = OUTPUT_DIR / "output_test.mp4"

def check_if_path_exists(path):
    if os.path.exists(path):
        print(f"{path} EXISTS!")
    else:
        print(f"{path} DOES NOT EXIST!")

check_if_path_exists(INPUT_FILE)

cmd = [
    "ffmpeg", "-y",
    "-i", INPUT_FILE,
    "-vf", "scale=48:27",
    "-c:v", "libx264",
    OUTPUT_FILE
]

subprocess.run(cmd, capture_output=True, text=True)

C:\Dev\AMVerge\backend\notebooks\ffmpeg_practice\input\naruto_clip.mp4 EXISTS!


CompletedProcess(args=['ffmpeg', '-y', '-i', WindowsPath('C:/Dev/AMVerge/backend/notebooks/ffmpeg_practice/input/naruto_clip.mp4'), '-vf', 'scale=48:27', '-c:v', 'libx264', WindowsPath('C:/Dev/AMVerge/backend/notebooks/ffmpeg_practice/output/output_test.mp4')], returncode=3752568763, stdout='', stderr="ffmpeg version 8.1.1-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers\n  built with gcc 15.2.0 (Rev13, Built by MSYS2 project)\n  configuration: --enable-gpl --enable-version3 --enable-shared --disable-w32threads --disable-autodetect --enable-cairo --enable-fontconfig --enable-iconv --enable-gnutls --enable-lcms2 --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-libsnappy --enable-zlib --enable-librist --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-libbluray --enable-libcaca --enable-libdvdnav --enable-libdvdread --enable-sdl2 --enable-libaribb24 --enable-libaribcaption --enable-libdav1d --enable-libdavs2 --enable-libopen

### Cutting and Merging clips
Here we cut clips using -ss and -to flags which trims from 'ss' value and 'to' value. Accepted format is seconds like 2.343, or time format such as HH:MM:SS for the timestamps on 'ss' and 'to' values.

We can also merge clips together by writing all output videos to an output txt file in the format:
```
file 'video_1_path'
file 'video_2_path'
```

we then feed the txt file that contains this information into FFmpeg via "-i", and it'll know how to parse it as long as the "-f" flag is present beforehand. Remember, FFmpeg works in steps from top to bottom.

In [21]:
# cutting clips
start_end_timestamps = [(1.2, 3.5), (8.6, 10.3)]

concat_file = OUTPUT_DIR / "clips.txt"

def convert_file_to_concat(file_name):
    return f"file '{file_name}'"

if os.path.exists(concat_file):
    os.remove(concat_file)

for i, timestamp in enumerate(start_end_timestamps):
    print(f"i: {i}, timestamp: {timestamp}")
    output_file = OUTPUT_DIR / f"{output_filename}_{i}.mp4"

    with open(concat_file, "a") as f:
        f.write(convert_file_to_concat(output_file) + "\n")
        
    cmd = [
        "ffmpeg", "-y",
        "-i", input_path,
        "-ss", str(timestamp[0]),
        "-to", str(timestamp[-1]),
        f"output/{output_file}"
    ]
    subprocess.run(cmd)

i: 0, timestamp: (1.2, 3.5)
i: 1, timestamp: (8.6, 10.3)


In [22]:
# merging clips

if os.path.exists(concat_file):
    with open(concat_file, "r") as f:
        print(f.read())
else:
    print(f"os path does not exist.")
print(f"output_path: {output_path}.mp4")

print(f"os.getcwd(): {os.getcwd()}")

cmd = [
    "ffmpeg",
    "-y",

    "-f", "concat",
    "-safe", "0",

    "-i", concat_file,

    "-c:v", "libx264",
    "-preset", "fast",
    "-crf", "18",

    "-c:a", "aac",

    f"{output_path}.mp4"
]

result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stderr)
# if os.path.exists(concat_file):
#     os.remove(concat_file)

file 'C:\Dev\AMVerge\backend\notebooks\ffmpeg_practice\output\output_naruto_clip_0.mp4'
file 'C:\Dev\AMVerge\backend\notebooks\ffmpeg_practice\output\output_naruto_clip_1.mp4'



NameError: name 'output_path' is not defined

### Decoding Video into Frames
Here we are practing decoding videos as well as creating the first version of TransNetV2 AI Scene detection. Videos files are basically streams of audio and video (which are streams of images). FFmpeg basically takes these videos, a list of instructions, then performs modifications to the video accordingly.

When we do a basic cmd conversion as above, we're going from CODEC -> decode -> raw frames -> encode -> output.

However, if we want to decode properly to feed into TransNetV2, we need to add -pix_fmt rgb24 which will convert the YUV frames to RGB frames if they aren't already (which is what TransNetV2 uses)

The stdout of the ffmpeg command is the video in bytes, we interpret it as uint8 using numpy to get the raw RGB values, then we reshape it into the sizes.

After everything, we're left with a numpy array of all the frames as [[R,G,B], [R,G,B]...[R,G,B]] to feed into TransNetV2.

In [ ]:
from pathlib import Path
import subprocess
import os
import numpy as np
BASE_DIR = Path.cwd().resolve()
print(f"BASE_DIR: {BASE_DIR}")

INPUT_FILE = BASE_DIR / "input/naruto_clip.mp4"
print(f"INPUT FILE: {INPUT_FILE}")

FRAME_WIDTH = 48
FRAME_HEIGHT = 27
FRAME_CHANNELS = 3

# one frame in a string of bytes is width* height * channel, so we make this helper function for that
def calculate_frame_bytes(width, height, channels):
    return width * height * channels

def check_if_path_exists(path):
    if os.path.exists(path):
        print(f"{path} EXISTS!")
    else:
        print(f"{path} does not exist!")

check_if_path_exists(INPUT_FILE)

cmd = [
    "ffmpeg", "-y",
    "-i", INPUT_FILE,
    "-pix_fmt", "rgb24",
    "-vf", "scale=48:27",
    "-f", "rawvideo",
    "pipe:1"
]

result = subprocess.run(cmd, capture_output=True)

print(f"TYPE OF STDOUT: {type(result.stdout)}")
print(f"LENGTH OF STDOUT: {len(result.stdout)}")


raw_video_bytes = result.stdout
converted_video_bytes = np.frombuffer(raw_video_bytes, dtype=np.uint8)
single_frame_byte_size = calculate_frame_bytes(FRAME_WIDTH, FRAME_HEIGHT, FRAME_CHANNELS)
num_frames = len(raw_video_bytes) // single_frame_byte_size
frames = np.reshape(converted_video_bytes, (num_frames, FRAME_HEIGHT, FRAME_WIDTH, FRAME_CHANNELS))
print(frames.shape)


BASE_DIR: C:\Dev\AMVerge\backend\notebooks\ffmpeg_practice
INPUT FILE: C:\Dev\AMVerge\backend\notebooks\ffmpeg_practice\input\naruto_clip.mp4
C:\Dev\AMVerge\backend\notebooks\ffmpeg_practice\input\naruto_clip.mp4 EXISTS!
TYPE OF STDOUT: <class 'bytes'>
LENGTH OF STDOUT: 2286144
(588, 27, 48, 3)


### AI Scene Detection

first iteration of AI scene detection

In [ ]:
import torch
from transnetv2_pytorch import TransNetV2
import transnetv2_pytorch
print(dir(transnetv2_pytorch))

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.is_available())

def convert_scenes_to_timestamps(src_fps, scenes):
    fps = 23.976

    cuts = scenes[:-1, 1]
    timestamps = cuts / fps

    return timestamps

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available else "cpu"

tensor = torch.from_numpy(frames).unsqueeze(0).to(device)

model = TransNetV2(device=device)
model.eval()

with torch.no_grad():
    single_frame_pred, all_frames_pred = model(tensor)
single_frame_pred = single_frame_pred.detach().cpu().numpy().squeeze()
scenes = model.predictions_to_scenes(single_frame_pred)

fps = 23.976

cuts = scenes[:-1, 1]
timestamps = cuts / fps

print(timestamps)

['TransNetV2', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'transnetv2_pytorch']
2.12.0+cu126
12.6
True
True
NVIDIA GeForce RTX 4070 SUPER
[ 2.96129463  5.96429763 10.96930264 13.97230564 19.22756089]


### Putting everything together
Below is the first iteration of the scene detection where I can properly benchmark the model.

In [ ]:
import torch
from transnetv2_pytorch import TransNetV2
from pathlib import Path
import subprocess
import os
import numpy as np

BASE_DIR = Path.cwd().resolve()

FRAME_WIDTH = 48
FRAME_HEIGHT = 27
FRAME_CHANNELS = 3

def calculate_frame_bytes(width, height, channels):
    return width * height * channels

def decode_video(input_video):
    check_if_path_exists(input_video)

    cmd = [
        "ffmpeg", "-y",
        "-i", str(input_video),
        "-pix_fmt", "rgb24",
        "-vf", "scale=48:27",
        "-f", "rawvideo",
        "pipe:1"
    ]
    result = subprocess.run(cmd, capture_output=True)

    raw_video_bytes = result.stdout
    video_bytes_as_rgb = np.frombuffer(raw_video_bytes, dtype=np.uint8)
    single_frame_byte_size = calculate_frame_bytes(FRAME_WIDTH, FRAME_HEIGHT, FRAME_CHANNELS)
    num_of_frames = len(raw_video_bytes) // single_frame_byte_size
    frames = np.reshape(video_bytes_as_rgb, (num_of_frames, FRAME_HEIGHT, FRAME_WIDTH, FRAME_CHANNELS))

    return frames

def convert_scenes_to_timestamps(src_fps, scenes):
    cuts = scenes[:-1, 1]
    timestamps = cuts / src_fps
    return timestamps, cuts

def get_fps(input_video):
    cmd = [
            "ffprobe",
            "-v", "error",
            "-select_streams", "v:0",
            "-show_entries", "stream=r_frame_rate",
            "-of", "default=noprint_wrappers=1:nokey=1",
            input_video
        ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    fps_str = result.stdout.strip()
    num, den = map(int, fps_str.split("/"))
    fps = num / den

    return fps

def run_model(frames):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tensor = torch.from_numpy(frames).unsqueeze(dim=0).to(device)

    model = TransNetV2(device=device)
    model.eval()

    single_frame_pred, all_frames_pred = model(tensor)
    single_frame_pred = single_frame_pred.detach().cpu().numpy().squeeze()
    scenes = model.predictions_to_scenes(single_frame_pred)

    fps = get_fps(INPUT_VIDEO)
    second_timestamps, frame_timestamps = convert_scenes_to_timestamps(fps, scenes)

    return second_timestamps, frame_timestamps

def benchmark_output(frame_timestamps, truth_file):

    i = 0
    matching = 0
    true_positives = []
    false_positives = []
    false_negatives = []
    truth_cuts = []

    with open (truth_file, "r") as f:
        for line in f:
            true_cut = int(line.strip())
            
            if true_cut in frame_timestamps:
                true_positives.append(true_cut)
                matching += 1

            # false_negative
            if true_cut not in frame_timestamps:
                false_negatives.append(true_cut)
            truth_cuts.append(true_cut) 

    for predicted_cut in frame_timestamps:
        if predicted_cut not in truth_cuts:
            false_positives.append(int(predicted_cut))

    print(f"TRUTH CUTS:\n{truth_cuts}\n")
    print(f"Cuts that were correctly identified: \t{true_positives}")
    print(f"Cuts that were incorrectly applied: \t{false_positives}")
    print(f"Cuts that were missed: \t\t\t{false_negatives}")    

INPUT_VIDEO = BASE_DIR / "input" / "franxx_ep.mkv"
truth_file = BASE_DIR / "truth_files/takopi_clip_1_truth.txt"

def check_if_path_exists(path):
    if not os.path.exists(path):
        raise Exception(f"ERROR: {INPUT_VIDEO} does not exist!")
    
frames = decode_video(INPUT_VIDEO)
second_timestamps, frame_timestamps = run_model(frames)

print(f"PREDICTIONS: \n{frame_timestamps}")

# benchmark_output(frame_timestamps, truth_file)

## TODO: add batches and improvements

In [ ]:
import torch

print(torch.__version__)
print(torch.version.cuda)

2.12.0+cu126
12.6
